# 화장품 HS 분류 베이스라인 (TF-IDF + LinearSVC)

이 노트북은 **코드만 포함**합니다. 학습 데이터와 결과 모델은 Google Drive에 별도로 저장합니다.

1. Google Drive에 `완제품 모델 테스트셋.xlsx`를 업로드합니다.
2. 아래 셀을 위에서부터 실행합니다.
3. 교차검증 결과와 학습된 모델이 Drive 폴더에 저장됩니다.

> 데이터 시트: `사례_중복제거` (196건). `판정요지`는 입력 변수에서 제외합니다. 현재 파일의 해당 시트에는 `사례ID`가 없어 그룹 단위 교차검증은 수행하지 않습니다.

In [1]:
!pip -q install pandas openpyxl scikit-learn joblib

In [ ]:
import pandas as pd

SHEET_ID = "13r9GlwDjXpoyyDAQy-TAQf4RppSPUcqEah2fkU6K_w4"
SHEET_NAME = "사례_중복제거"

DATA_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

df = pd.read_csv(DATA_URL)

print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

SHEET_NAME = '사례_중복제거'
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, dtype={'결정세번': str})

required = ['결정세번', '품명', '주요성분', '물품형태·포장', '용도', '제품유형']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'필수 열이 없습니다: {missing}')

df = df[required].copy()
df['결정세번'] = df['결정세번'].astype(str).str.strip()
df = df.dropna(subset=['결정세번'])
for c in required[1:]:
    df[c] = df[c].fillna('').astype(str).str.lower().str.strip()

print('데이터 크기:', df.shape)
print('고유 HS 코드 수:', df['결정세번'].nunique())
print('주요성분 누락:', (df['주요성분'] == '').sum())
print(df['결정세번'].value_counts().head(10))

In [ ]:
from sklearn.compose import ColumnTransformer

X = df[['품명', '주요성분', '물품형태·포장', '용도', '제품유형']].copy()
y = df['결정세번'].to_numpy()
char = lambda: TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=1, sublinear_tf=True)
features = ColumnTransformer([
    ('품명_char', char(), '품명'),
    ('품명_word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, sublinear_tf=True, token_pattern=r'(?u)\b\w+\b'), '품명'),
    ('성분_char', char(), '주요성분'),
    ('형태_char', char(), '물품형태·포장'),
    ('용도_char', char(), '용도'),
    ('제품유형_char', char(), '제품유형')
], remainder='drop')

model = Pipeline([('tfidf', features), ('clf', LinearSVC(class_weight='balanced', C=1.0, random_state=42))])
print('모델 구성 완료')

## 교차검증

2-Fold Stratified Cross-Validation을 수행합니다. 2건 미만인 HS 코드는 독립적인 교차검증 평가가 불가능하므로 지표 계산에서 제외합니다. 단, 모델 학습 데이터에는 포함됩니다.

※ 현재 데이터 시트에 사례ID가 없으므로 동일 사례에서 파생된 행이 fold 간에 분리될 가능성을 별도로 확인해야 합니다.

In [ ]:
class_counts = pd.Series(y).value_counts()
evaluable = set(class_counts[class_counts >= 2].index)
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
rows = []

for fold, (train_idx, test_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
    fold_model = clone(model)
    fold_model.fit(X.iloc[train_idx], y[train_idx])
    eval_idx = [i for i in test_idx if y[i] in evaluable]
    if not eval_idx:
        continue
    pred = fold_model.predict(X.iloc[eval_idx])
    scores = fold_model.decision_function(X.iloc[eval_idx])
    classes = fold_model.named_steps['clf'].classes_
    top3 = []
    for score in scores:
        order = np.argsort(score)[::-1][:3]
        top3.append([classes[j] for j in order])
    for idx, p, candidates in zip(eval_idx, pred, top3):
        rows.append({'원본행번호': idx, '실제_HS': y[idx], '예측_HS': p, 'Top3_HS': ', '.join(candidates), 'Top3_정답포함': y[idx] in candidates, 'Fold': fold})

pred_df = pd.DataFrame(rows)
top1 = accuracy_score(pred_df['실제_HS'], pred_df['예측_HS'])
top3 = pred_df['Top3_정답포함'].mean()
macro = f1_score(pred_df['실제_HS'], pred_df['예측_HS'], average='macro', zero_division=0)
weighted = f1_score(pred_df['실제_HS'], pred_df['예측_HS'], average='weighted', zero_division=0)
summary = pd.DataFrame([{'평가건수': len(pred_df), '평가가능_HS코드수': len(evaluable), 'Top1_Accuracy': top1, 'Top3_Accuracy': top3, 'Macro_F1': macro, 'Weighted_F1': weighted}])
display(summary)
display(pred_df.head())

with pd.ExcelWriter(BASE_DIR / 'baseline_cv_results.xlsx') as writer:
    summary.to_excel(writer, sheet_name='평가요약', index=False)
    pred_df.to_excel(writer, sheet_name='교차검증_예측', index=False)

print('평가 결과 저장:', BASE_DIR / 'baseline_cv_results.xlsx')

## 전체 데이터로 최종 모델 학습 및 저장

교차검증이 끝나면 전체 196건으로 최종 모델을 학습합니다. 저장된 모델은 이 노트북과 별도 파일입니다.

In [ ]:
final_model = clone(model)
final_model.fit(X, y)
MODEL_PATH = BASE_DIR / 'cosmetics_hs_tfidf_linearsvc.joblib'
joblib.dump(final_model, MODEL_PATH, compress=3)
print('최종 모델 저장:', MODEL_PATH)
print('학습 데이터 건수:', len(X))
print('학습 HS 코드 수:', len(final_model.named_steps['clf'].classes_))

## 새 제품 예측

제품 정보를 입력하고 셀을 실행하면 상위 3개 HS 코드 후보와 결정 점수를 출력합니다. 결정 점수는 확률이 아닙니다.

In [ ]:
def predict_hs(product_name, ingredients='', form='', purpose='', product_type=''):
    row = pd.DataFrame([{'품명': product_name.lower().strip(), '주요성분': ingredients.lower().strip(), '물품형태·포장': form.lower().strip(), '용도': purpose.lower().strip(), '제품유형': product_type.lower().strip()}])
    scores = final_model.decision_function(row)[0]
    classes = final_model.named_steps['clf'].classes_
    order = np.argsort(scores)[::-1][:3]
    return pd.DataFrame([{'순위': rank, 'HS코드': classes[i], '결정점수': float(scores[i])} for rank, i in enumerate(order, start=1)])

predict_hs(product_name='Moisturizing Cream', ingredients='Glycerin, Niacinamide', form='크림', purpose='피부 보습', product_type='기초화장품')